# Singapore Smart City: Level 2 Diagnostic AI
## Vision-Language Model (VLM) Zero-Shot Anomaly Captioning

Traditional anomaly detection systems only output a boolean flag (`Anomaly=True`). This is not immediately actionable for traffic controllers.

In this **Level 2 Diagnostic Pipeline**, when the Level 1 model (YOLO) detects an anomaly (e.g., 0 cars moving on a major highway), we pass the raw camera frame to a lightweight, state-of-the-art **Vision-Language Model (Microsoft Florence-2)**.

The VLM generates a **Zero-Shot Natural Language Caption** explaining *why* the anomaly occurred (e.g., "A red truck has collided with the center divider").

### Execution Environment
> **Cost:** $0, runs freely on Colab/Kaggle T4 GPUs.

In [ ]:
!pip install -q transformers einops flash_attn accelerate pillow

In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

### 1. Load the Microsoft Florence-2 VLM
Florence-2 is an incredibly capable open-source VLM that is small enough (0.7B parameters) to run blazingly fast on free GPUs, but powerful enough to generate highly detailed scene descriptions.

In [ ]:
model_id = "microsoft/Florence-2-large"

print(f"Loading {model_id} to {device}...")
processor = AutoProcessor.prompt_mode(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch_dtype, trust_remote_code=True).to(device)

### 2. The Anomaly Captioning Pipeline
We define a function that takes an anomalous image and prompts the VLM to describe the failure condition strictly.

In [ ]:
def generate_anomaly_report(image_path):
    """
    Accepts an image path that was flagged as anomalous by Level 1/Level 2,
    and generates a natural language diagnostic report.
    """
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return f"[Error] Could not load image: {e}"
        
    # We use <DETAILED_CAPTION> to force the VLM to explain the scene thoroughly.
    task_prompt = "<DETAILED_CAPTION>"
    
    inputs = processor(text=task_prompt, images=image, return_tensors="pt").to(device, torch_dtype)
    
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3,
        do_sample=False
    )
    
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    
    # Clean the output tags
    parsed_answer = processor.post_process_generation(
        generated_text, 
        task=task_prompt, 
        image_size=(image.width, image.height)
    )
    
    return parsed_answer[task_prompt]

print("✅ VLM Anomaly Diagnostic Pipeline Initialized.")

In [ ]:
# Example Execution (Uncomment when running in Colab with actual data)
"""
flagged_image = '/content/drive/MyDrive/sg_smart_city/data/raw/2026-03-09/1001/11-56-20.jpg'
report = generate_anomaly_report(flagged_image)

print(f"[LTA Camera 1001 Anomaly Alert]")
print(f"VLM Diagnostic: {report}")
"""